In [1]:
# import required python modules
import os
import zipfile
import pandas as pd

In [1]:
# set input and output paths
base_dir = r"D:\Egypt-Opportunity-Radar-Task3\data\raw\production"
zip_path = os.path.join(base_dir, "task3_unido_iip_egypt_export.zip")
out_dir = os.path.join(base_dir, "cleaned")
os.makedirs(out_dir, exist_ok=True)

In [1]:
# open raw zip archive
with zipfile.ZipFile(zip_path, "r") as zf:
    names = zf.namelist()
    tables = {name: pd.read_csv(zf.open(name)) for name in names}

In [1]:
# clean main index table
df = tables["data.csv"].copy()
df.columns = [c.strip() for c in df.columns]
df = df.dropna(how="all").drop_duplicates()

In [1]:
# convert value column to numeric
df["Value"] = pd.to_numeric(df["Value"], errors="coerce")
df = df.sort_values(["VariableCode", "ActivityCode", "Year"]).reset_index(drop=True)

In [1]:
# save cleaned main csv
csv_out = os.path.join(out_dir, "cleaned_task3_unido_iip_egypt_export.csv")
df.to_csv(csv_out, index=False, encoding="utf-8-sig")

In [1]:
# save all zip tables
xlsx_out = os.path.join(out_dir, "cleaned_task3_unido_iip_egypt_export.xlsx")
with pd.ExcelWriter(xlsx_out, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="data", index=False)
    for name, tbl in tables.items():
        if name != "data.csv":
            sheet = name.replace(".csv", "")[:31]
            tbl.dropna(how="all").to_excel(writer, sheet_name=sheet, index=False)